# AI6130 Assignment 2 - TEST RUN (10-15 min)

**Purpose:** Verify all configurations work before full run

Tests:
- 2 models (OpenLLaMA, TinyLlama)
- 2 adapters (LoRA, AdapterP) with CORRECT parameters
- Minimal training (1 step each)
- Quick evaluation

**Run this FIRST to verify everything works!**


In [ ]:
import os
import time
import pandas as pd
from datetime import datetime

print("="*80)
print("TEST RUN - Assignment 2")
print("="*80)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# STEP 1: Setup
print("[1/6] Setup environment...")
os.chdir('/kaggle/working')

if not os.path.exists('AI6130_Assignment2'):
    os.system('git clone https://github.com/duyngtr16061999/AI6130_Assignment2')

os.chdir('AI6130_Assignment2')

os.system('pip install -q fire datasets')
os.system('pip install -q transformers==4.38.0 accelerate==0.28.0')
os.system('pip uninstall -q -y peft')  # Use local peft from repo

os.environ["WANDB_MODE"] = "disabled"
print("  [OK]")
print()

# STEP 2: Enable KV cache
print("[2/6] Enable KV cache...")
os.system("sed -i 's/use_cache=False/use_cache=True/g' evaluate.py")
print("  [OK]")
print()

# STEP 3: Clean models
print("[3/6] Clean models directory...")
if os.path.exists('./trained_models'):
    import shutil
    shutil.rmtree('./trained_models')
os.makedirs('./trained_models', exist_ok=True)
print("  [OK]")
print()

# STEP 4: TEST TRAINING
print("[4/6] TEST TRAINING (1 step per model)...")
print()

# CORRECT adapter configuration based on repo README
models = [
    ("openlm-research/open_llama_3b_v2", "OpenLLaMA"),
    ("TinyLlama/TinyLlama_v1.1", "TinyLlama")
]

# Training config: (adapter_name, extra_flags, display_name)
adapters_train = [
    ("lora", "", "LoRA"),
    ("bottleneck", "--use_adapterp", "AdapterP")  # CORRECT!
]

# Evaluation config: (adapter_name, display_name)
adapters_eval = [
    ("LoRA", "LoRA"),
    ("AdapterP", "AdapterP")
]

test_results = []
num = 0
total = len(models) * len(adapters_train)

for model_path, model_name in models:
    for (adapter_name, extra_flags, adapter_display) in adapters_train:
        num += 1
        out_dir = f"./trained_models/{model_name}-{adapter_display}-test"

        print(f"[{num}/{total}] {model_name} + {adapter_display}")

        # Build training command with correct flags
        train_cmd = (
            f"python finetune.py --base_model '{model_path}' "
            f"--data_path './ft-training_set/math_7k.json' "
            f"--output_dir '{out_dir}' --batch_size 4 --micro_batch_size 4 "
            f"--num_epochs 1 --learning_rate 3e-4 --cutoff_len 256 "
            f"--val_set_size 120 --adapter_name {adapter_name} {extra_flags} "
            f"--max_steps 1"
        )

        start = time.time()
        result = os.system(train_cmd)
        elapsed = time.time() - start

        status = "SUCCESS" if result == 0 else "FAILED"
        print(f"  [{'OK' if result == 0 else 'FAIL'}] {status} ({elapsed:.1f}s)\n")

        test_results.append({
            'Model': f"{model_name}-{adapter_display}",
            'Status': status,
            'Time': f"{elapsed:.1f}s"
        })

# STEP 5: TEST EVALUATION
print("[5/6] TEST EVALUATION (quick check)...")
print()

def is_valid(model_dir):
    if not os.path.exists(model_dir):
        return False
    for f in ['adapter_config.json', 'adapter_model.bin']:
        fp = os.path.join(model_dir, f)
        if not os.path.exists(fp) or os.path.getsize(fp) == 0:
            return False
    return True

eval_results = []
idx = 0

for model_path, model_name in models:
    for (eval_name, adapter_display) in adapters_eval:
        out_dir = f"./trained_models/{model_name}-{adapter_display}-test"
        idx += 1

        if not is_valid(out_dir):
            print(f"  [{idx}] {model_name}-{adapter_display}: Skipped (invalid)")
            eval_results.append({'Model': f"{model_name}-{adapter_display}", 'Status': 'SKIPPED'})
            continue

        print(f"  [{idx}] {model_name}-{adapter_display} on AddSub")
        eval_cmd = (
            f"python evaluate.py --adapter {eval_name} "
            f"--dataset AddSub --base_model '{model_path}' "
            f"--lora_weights '{out_dir}' > /dev/null 2>&1"
        )

        result = os.system(eval_cmd)
        status = "SUCCESS" if result == 0 else "FAILED"
        print(f"      [{'OK' if result == 0 else 'FAIL'}] {status}")

        eval_results.append({'Model': f"{model_name}-{adapter_display}", 'Status': status})

# STEP 6: RESULTS
print()
print("="*80)
print("[6/6] TEST RESULTS")
print("="*80)
print()

train_df = pd.DataFrame(test_results)
print("Training:")
print(train_df.to_string(index=False))
print()

eval_df = pd.DataFrame(eval_results)
print("Evaluation:")
print(eval_df.to_string(index=False))
print()

train_ok = all(r['Status'] == 'SUCCESS' for r in test_results)
eval_ok = all(r['Status'] in ['SUCCESS', 'SKIPPED'] for r in eval_results)

if train_ok and eval_ok:
    print("[OK] ALL TESTS PASSED!")
    print("[OK] Ready for FULL run")
else:
    print("[WARN] SOME TESTS FAILED")
    print("[WARN] Check errors above")

print()
print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
